Importo las librerías para trabajar

In [1]:
import pandas as pd
import requests
import shutil
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
from scipy import sparse
from surprise import Dataset, Reader
from surprise import SVD
from surprise import accuracy
from surprise.model_selection import train_test_split
import gradio as gr

In [2]:
dfml=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/ML_ETL_plataformas.csv')

In [3]:
dfmlr1=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/1.csv')
dfmlr2=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/2.csv')
dfmlr3=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/3.csv')
dfmlr4=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/4.csv')
dfmlr5=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/5.csv')
dfmlr6=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/6.csv')
dfmlr7=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/7.csv')
dfmlr8=pd.read_csv(r'/Users/herminiaaguirre/Desktop/Henry/Proyectos varios/4.PI/MLOps/datasets/ratings/8.csv')

In [4]:
dfmlr=(pd.concat([dfmlr1,dfmlr2,dfmlr3,dfmlr4,dfmlr5,dfmlr6,dfmlr7,dfmlr8], axis=0))

Elimino la columna 'timespan' de df_users porque no la vamos a utilizar

In [5]:
dfmlr = dfmlr.drop(['timestamp'], axis=1)

In [6]:
dfml.head(1)

,id,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,plataforma,duration_int,duration_type,score
0,as1,s1,movie,the grand seduction,don mckellar,"brendan gleeson, taylor kitsch, gordon pinsent",canada,2021-03-30 00:00:00,2014,G,113 min,"comedy, drama",a small fishing village must procure a local d...,amazon,113,min,3.549127


Elimino columnas no relevantes para el análisis

In [7]:
dfml= dfml.drop(['show_id','director','cast','country','date_added','release_year','rating','duration','description','duration_int','duration_type','score'], axis=1)

In [8]:
dfml.head(1)

,id,type,title,listed_in,plataforma
0,as1,movie,the grand seduction,"comedy, drama",amazon


Cambio de nombre de la columna 'id' a 'movieId'

In [9]:
dfml = dfml.rename(columns={'id': 'movieId'})
# We have now the same colum 'movieId' for both dfs

In [10]:
dfml.head(2)

,movieId,type,title,listed_in,plataforma
0,as1,movie,the grand seduction,"comedy, drama",amazon
1,as2,movie,take care good night,"drama, international",amazon


Uno los dos df

In [11]:
dfcompl = pd.merge(left=dfmlr,right=dfml)

In [12]:
dfcompl.shape
# we have 11024289 rows 

(11024289, 7)

In [13]:
dfcompl.head(5)

,userId,rating,movieId,type,title,listed_in,plataforma
0,1,1.0,as680,tv show,the english civil war,"documentary, special interest",amazon
1,583,4.5,as680,tv show,the english civil war,"documentary, special interest",amazon
2,765,5.0,as680,tv show,the english civil war,"documentary, special interest",amazon
3,2116,3.0,as680,tv show,the english civil war,"documentary, special interest",amazon
4,2143,3.0,as680,tv show,the english civil war,"documentary, special interest",amazon


Reordeno la variable de objetivo

In [14]:
columnas = dfcompl.columns.tolist()
columnas = ['userId', 'movieId', 'type', 'title', 'listed_in', 'plataforma', 'rating']
dfcompl = dfcompl[columnas]
dfcompl.head(2)

,userId,movieId,type,title,listed_in,plataforma,rating
0,1,as680,tv show,the english civil war,"documentary, special interest",amazon,1.0
1,583,as680,tv show,the english civil war,"documentary, special interest",amazon,4.5


Realizo una copia de dfcompl como mejor práctica

In [15]:
df_ml = dfcompl.copy()
df_ml.sample(3)

,userId,movieId,type,title,listed_in,plataforma,rating
5427969,66440,ns3246,tv show,singapore social,"international tv shows, reality tv",netflix,4.0
8335965,6201,as1682,tv show,michael jackson: the ultimate icon,tv shows,amazon,5.0
2357561,436,ns7801,tv show,psiconautas,"international tv shows, spanish-language tv sh...",netflix,2.5


Creo una lista de valores de la columna 'listed_in'

In [17]:
df_ml['listed_in'] = df_ml.listed_in.str.split(' , ')

Rango de verificación de los valores de 'rating'

In [18]:
print(df_ml['rating'].min())
print(df_ml['rating'].max())
# rating values goes between of 0.5 - 5

0.5
5.0


Preparo el df con la columna 'userId' 'movieId' 'rating'

In [19]:
df_prepared = df_ml.loc[:,["userId","movieId","rating"]]

Utilizo una muestra de 1M filas

In [20]:
df_muest = df_prepared.sample(n=1000000, replace=True)

In [21]:
df_muest.head()

,userId,movieId,rating
2824810,32442,as8972,0.5
2196932,19837,as4089,5.0
10099237,10911,as2630,1.0
1100215,60078,ns968,4.0
6474929,28855,as7240,3.0


Modelado

Realizo una codificación de la columna movieId a int

In [22]:
col = df_muest.movieId

# creamos objeto labelencoder y ajustamos la columna
labelE = LabelEncoder()
labelE.fit(col)

#tranformamos la columna
col_transformada = labelE.transform(col)

#lo reemplazamos a show id original
df_muest.movieId = col_transformada
df_muest.head()

,userId,movieId,rating
2824810,32442,8858,0.5
2196932,19837,3433,5.0
10099237,10911,1813,1.0
1100215,60078,22963,4.0
6474929,28855,6935,3.0


In [ ]:
# df_prepared = df_ml.loc[:,["userId","movieId","rating"]]

In [ ]:
# df_prepared.head()

In [ ]:
# from scipy.sparse import csr_matrix

# sparse_matrix = csr_matrix(df_prepared.values)

Convierto los df a surprise datasets

In [23]:
reader = Reader(line_format='user item rating', rating_scale=(0.5, 5))
data = Dataset.load_from_df(df_muest, reader)

In [24]:
# Build full trainset
data_train_surp = data.build_full_trainset()

# Define the model
svd = SVD()

# Train the model
svd.fit(data_train_surp)

In [25]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [26]:
predictions = svd.test(testset)

In [27]:
predictions

[Prediction(uid=119410, iid=11811, r_ui=3.0, est=3.3035782375691567, details={'was_impossible': False}),
 Prediction(uid=54005, iid=4396, r_ui=4.5, est=3.7272781442072405, details={'was_impossible': False}),
 Prediction(uid=12306, iid=13250, r_ui=3.0, est=3.1299017664188535, details={'was_impossible': False}),
 Prediction(uid=12962, iid=22181, r_ui=3.0, est=2.955067219335197, details={'was_impossible': False}),
 Prediction(uid=115715, iid=11436, r_ui=4.0, est=3.917206217232953, details={'was_impossible': False}),
 Prediction(uid=33940, iid=12026, r_ui=3.5, est=2.784460490879487, details={'was_impossible': False}),
 Prediction(uid=25086, iid=20899, r_ui=4.0, est=3.9172474131200636, details={'was_impossible': False}),
 Prediction(uid=113525, iid=10627, r_ui=3.0, est=3.674915887921005, details={'was_impossible': False}),
 Prediction(uid=30394, iid=5692, r_ui=3.0, est=3.5374337407594316, details={'was_impossible': False}),
 Prediction(uid=260233, iid=3620, r_ui=3.0, est=3.3280731606932927,

In [46]:
svd.predict(356,567,4)

Prediction(uid=356, iid=567, r_ui=4, est=3.729538794417126, details={'was_impossible': False})